# RRT (rapidly exploring random trees) vs RRT*

In [1]:
import io
import copy
import math
import random
import imageio
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely.geometry import LineString
from shapely.geometry import Point as ShapelyPoint
from shapely.geometry import Polygon as ShapelyPolygon

Firstly, the Node class is defined to represent a single node of the tree structure.
Each node stores:

* x and y: the coordinates of the node;

* parent: a reference to the parent node, initialized as None by default.

The parent attribute is used for reconstructing the tree and tracing the final path from a node back to the root.

In [2]:
class Node:
    def __init__(self, x, y, parent = None):
        self.x = x
        self.y = y
        self.parent = parent

Then, some basic functions are defined to implement the RRT algorithm. These include:

* distance (q, qrand): computes the euclidean distance between two nodes;

* find_qnear(tree, qrand): finds the nearest node of the tree given the random generated node qrand. The function iterates through all the nodes in the tree, computes their distance from qrand, and updates the nearest node whenever a smaller distance is found. The function returns the nearest node qnear.

* verify_segment_collision(qnear, qs): checks whether the segment connecting qnear and qs intersects any obstacle in the environment. The function returns True if a collision is detected and False otherwise.

* find_qs(qnear, qrand): determines the farthest reachable point qs along the segment joining qnear and qrand while maintaining a collision-free path.
The segment is discretized into intermediate points, and the algorithm progressively checks for collisions. The exploration stops when a collision is found, while the last valid point is stored as qs. The returned node is connected to qnear through the parent attribute.

* get_final_path(tree): reconstructs the final path from the last node of the tree back to the root node by following the parent references.
The path is then reversed in order to obtain it from the start node to the goal node.

In [3]:
def distance(q, qrand):
    return math.sqrt((q.x - qrand.x)**2 + (q.y - qrand.y)**2)

def find_qnear(tree, qrand): #finds the nearest node of the tree with respect to qrand
  qnear = None
  min_distance = float('inf')
  for node in tree:
    if (distance(node, qrand)) < min_distance: #compares distances
      min_distance = distance(node, qrand)
      qnear = node
  return qnear

def verify_segment_collision(qnear, qs):
    segment = LineString([(qnear.x, qnear.y), (qs.x, qs.y)])
    for obs in OBSTACLES:
        if obs.intersects(segment):
            return True
    return False

def find_qs(qnear, qrand):
    segment_length = distance(qnear, qrand)
    if segment_length < 1e-6:
        return None
    steps = max(1, int(segment_length))
    qs = None
    for i in range(1, steps + 1):
        t = i / steps
        x = qnear.x + t * (qrand.x - qnear.x)
        y = qnear.y + t * (qrand.y - qnear.y)
        temp = Node(x, y)
        if verify_segment_collision(qnear, temp):
            break
        qs = Node(x, y, qnear)
    return qs

def get_final_path(tree):
    path, cur = [], tree[-1]
    while cur:
        path.append((cur.x, cur.y))
        cur = cur.parent
    return list(reversed(path))

The main procedure of the RRT algorithm is implemented through the function RRT(qstart, qgoal, iterations).

The function takes as input:

* qstart: the initial node;
* qgoal: the goal node;
* iterations: the maximum number of iterations, set by default to 1000.

Initially, the tree is composed only of the root node qstart. Then, at each iteration, a random node qrand is generated inside the configuration space. The nearest node of the tree, denoted as qnear, is found through the function find_qnear(tree, qrand) and the function find_qs(qnear, qrand) is used to compute the farthest collision-free node qs along the segment connecting qnear and qrand. If a valid node qs exists, it is added to the tree, allowing the exploration to progressively expand toward unexplored regions of the configuration space. Every 100 iterations, instead of generating a random node, the algorithm directly attempts to connect the tree to the goal node qgoal. If the segment connecting qnear and qgoal is collision-free, the goal node is added to the tree by setting its parent to qnear, and the algorithm terminates successfully. During the execution, copies of the tree are stored inside tree_list in order to keep track of the evolution of the algorithm at each iteration. The function finally returns tree_list, which contains all the intermediate states of the tree construction process.

In [4]:
#randomly generate a point qrand in the configuration space C
#find the closest node qnear of the three
#find the point qs on the segment joining qrand and qnear that maximizes the distance from qnear and under the costraint that the segment joining qnear and qs is collision-free

def RRT(qstart, qgoal, iterations=1000):
    tree = [qstart]
    tree_list = [tree.copy()]
    for i in range(iterations):
        if i % 100 != 0:
            qrand = Node(random.uniform(0, 100), random.uniform(0, 100))
            qnear = find_qnear(tree, qrand)
            qs = find_qs(qnear, qrand)
            if qs is not None:
                tree.append(qs)
        else:
            qnear = find_qnear(tree, qgoal)
            if not verify_segment_collision(qnear, qgoal):
                qgoal.parent = qnear
                tree.append(qgoal)
                tree_list.append(tree.copy())
                return tree_list
        tree_list.append(tree.copy())
    return tree_list


As before, some basic functions are defined to implement the RRT* algorithm.

* compute_cost(q): computes the cost of a node q with respect to the root node qstart. The cost is obtained by traversing the tree backwards through the parent references and summing the euclidean distances between each node and its parent. Therefore, the cost represents the total path length from the root to the node.


* choose_parent(tree, qs, qnear, epsilon): selects the best parent for the node qs. Initially, the parent of qs is assumed to be qnear. Then, all the nodes within a neighborhood of radius epsilon around qs are analyzed. For each neighboring node, the algorithm computes the cost of reaching qs through that node. If a lower-cost collision-free connection is found, the corresponding node becomes the new parent of qs.

* rewire(tree, qs, epsilon): the function checks whether connecting the neighboring nodes of qs through qs produces a lower path cost from the root. If a shorter collision-free path is found, the parent of the neighboring node is updated accordingly.

In [5]:
def compute_cost(q): #computes the cost of a given node considered as the distance from the node to qstart
    cost = 0
    current_node = q
    while current_node.parent is not None:
        cost += distance(current_node, current_node.parent)
        current_node = current_node.parent
    return cost

def choose_parent(tree, qs, qnear, epsilon):
    best_parent = qnear
    best_cost = compute_cost(qnear) + distance(qnear, qs)
    for node in tree:
        if distance(node, qs) <= epsilon:
            candidate_cost = compute_cost(node) + distance(node, qs)
            if candidate_cost < best_cost:
                if not verify_segment_collision(node, qs):
                    best_cost = candidate_cost
                    best_parent = node
    qs.parent = best_parent

def rewire(tree, qs, epsilon):
    qs_cost = compute_cost(qs)
    for node in tree:
        if distance(node, qs) <= epsilon:
            new_cost = qs_cost + distance(qs, node) #computes the new cost going by qs
            if new_cost < compute_cost(node) and not verify_segment_collision(qs, node):
                node.parent = qs

The main procedure of the RRT* algorithm is implemented through the function RRT(qstart, qgoal, iterations, epsilon).

The function takes as input:

* qstart: the initial node;


* qgoal: the goal node;


* iterations: the maximum number of iterations, set by default to 1000.


Initially, the tree is composed only of the root node qstart. Then, at each iteration, a random node qrand is generated inside the configuration space. The nearest node of the tree, denoted as qnear, is found through the function find_qnear(tree, qrand) and the function find_qs(qnear, qrand) is used to compute the farthest collision-free node qs along the segment connecting qnear and qrand. If a valid node qs is obtained, the choose_parent function selects the optimal parent within the neighborhood of radius epsilon, minimizing the path cost from the root. Once the best parent is assigned, qs is added to the tree, and the rewire procedure is applied to locally improve the tree structure by reconnecting neighboring nodes through qs if a lower-cost path is found. Every 100 iterations, instead of generating a random node, the algorithm directly attempts to connect the tree to the goal node qgoal. If the segment connecting qnear and qgoal is collision-free, the goal node is added to the tree by setting its parent to qnear, and the algorithm terminates successfully. During the execution, copies of the tree are stored inside tree_list in order to keep track of the evolution of the algorithm at each iteration. The function finally returns tree_list, which contains all the intermediate states of the tree construction process.

In [28]:
def RRT_star(qstart, qgoal, iterations=1000, epsilon=10):
    tree = [qstart]
    tree_list = [copy.deepcopy(tree)]
    for i in range(iterations):
        if i % 100 != 0:
            qrand = Node(random.uniform(0, 100), random.uniform(0, 100))
            qnear = find_qnear(tree, qrand)
            qs = find_qs(qnear, qrand)
            if qs is not None:
                choose_parent(tree, qs, qnear, epsilon)
                tree.append(qs)
                rewire(tree, qs, epsilon)
        else:
            qnear = find_qnear(tree, qgoal)
            if not verify_segment_collision(qnear, qgoal):
                qgoal.parent = qnear
                tree.append(qgoal)
                tree_list.append(copy.deepcopy(tree))
                return tree_list
        tree_list.append(copy.deepcopy(tree))
    return tree_list

This block defines the environment used for the simulation, including both the obstacle space and the visualization settings.

The obstacles are represented as a list of polygons, defined using Shapely geometric objects. These shapes define the constraints that the RRT and RRT* algorithms must avoid while constructing the tree.

In addition to the obstacle definition, a set of visualization parameters is specified:

* OBSTACLE_COLORS defines a color palette used to render each obstacle with a distinct visual style;


* TREE_COLOR_RRT and TREE_COLOR_RRTS define the colors used to visualize the tree generated by the RRT and RRT* algorithms respectively;



In [25]:
OBSTACLES = [
    ShapelyPolygon([(5,40),(20,35),(28,50),(18,62),(5,55)]),
    ShapelyPolygon([(35,15),(55,10),(60,28),(48,35),(30,28)]),
    ShapelyPolygon([(45,45),(65,40),(70,60),(55,68),(40,60)]),
    ShapelyPolygon([(20,70),(38,65),(42,80),(30,90),(15,85)]),
    ShapelyPolygon([(65,65),(82,60),(88,75),(78,88),(62,80)]),
    ShapelyPolygon([(50,25),(68,22),(72,38),(58,44),(45,35)]),
    ShapelyPolygon([(10,15),(28,10),(32,25),(20,32),(8,25)]),
    ShapelyPolygon([(75,40),(92,38),(95,52),(82,58),(70,50)]),
]

OBSTACLE_COLORS = ['#c0392b', '#d35400', '#27ae60', '#2980b9', '#8e44ad', '#16a085', '#f39c12', '#2c3e50']
TREE_COLOR_RRT = '#27ae60'
TREE_COLOR_RRTS = '#2980b9'

Then, a set of functions is introduced to visualize the environment and the execution of the algorithms.

* verify_point_collision(x, y): checks whether a point belongs to the free configuration space. The function creates a geometric point and verifies that it is not contained inside any obstacle. It returns True if the point is collision-free and False otherwise.


* draw_obstacles(ax): draws all the obstacles in the environment. For each polygonal obstacle, the coordinates of its vertices are extracted and the corresponding shape is rendered with the associated color.


* draw_panel(ax, tree_list, path, color, label, i, show_path): visualizes the state of the algorithm at a given iteration. The function displays the obstacles, the generated tree, the start and goal nodes, and optionally the final path. Tree edges are drawn by connecting each node to its parent, allowing the progressive expansion of the exploration tree to be visualized over time.

In [11]:
def verify_point_collision(x, y):
    p = ShapelyPoint(x, y)
    for obs in OBSTACLES:
        if obs.contains(p):
            return False
    return True

def draw_obstacles(ax):
    for obs, color in zip(OBSTACLES, OBSTACLE_COLORS):
        xs, ys = obs.exterior.xy #extracts the coordinates of each the vertex of the obstacles
        ax.fill(xs, ys, color = color)  #fills and draws the obstacle with the chosen colour

def draw_panel(ax, tree_list, path, color, label, i, show_path):
    tree = tree_list[min(i, len(tree_list)-1)]
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel('X', fontsize=9)
    ax.set_ylabel('Y', fontsize=9)
    ax.set_title(f'{label} iterazione: {min(i+1, len(tree_list))} / {len(tree_list)} nodi: {len(tree)}', fontsize=10, fontweight='bold', pad=8)
    draw_obstacles(ax)
    for node in tree:
        if node.parent:
            ax.plot([node.x, node.parent.x], [node.y, node.parent.y], color=color, linewidth=1.0) #disegna gli archi
    ax.plot(sx, sy, marker='*', color='#2ecc71', markersize=14, markeredgecolor='black', markeredgewidth=0.5, label='Start') #disegna qstart
    ax.plot(gx, gy, marker='*', color='#3498db', markersize=14, markeredgecolor='black', markeredgewidth=0.5, label='Goal') #disegna qgoal
    if show_path and len(path) > 1:
        px, py = zip(*path)
        ax.plot(px, py, color='#e74c3c', linewidth=2.8, alpha=0.95, zorder=5, label='Path')
    ax.legend(fontsize=8, loc='upper left')

This block executes both the RRT and RRT* algorithms and generates an animated visualization of their exploration process.

First, the start node (qstart) and the goal node (qgoal) are randomly generated in different regions of the configuration space. In both cases, the sampled points are accepted only if they belong to the free space, ensuring that neither node lies inside an obstacle. Once the start and goal configurations are generated, both algorithms are executed:

RRT(Node(sx, sy), Node(gx, gy))

RRT_star(Node(sx, sy), Node(gx, gy))

The corresponding final paths are then reconstructed through the function get_final_path. After the execution phase, the code creates an animated GIF comparing the evolution of the two algorithms over time. For each frame: two panels are displayed side by side; the left panel shows the evolution of the RRT algorithm; the right panel shows the evolution of the RRT* algorithm.

The trees are progressively drawn iteration after iteration, while the final path is displayed only in the last frame of the animation. Each generated frame is stored and appended to the GIF using the imageio library.


In [29]:
qstart = False
while not qstart:
    sx, sy = random.uniform(0, 25), random.uniform(0, 25)
    if verify_point_collision(sx, sy):
        qstart = True
qgoal = False
while not qgoal:
    gx, gy = random.uniform(75, 100), random.uniform(75, 100)
    if verify_point_collision(gx, gy):
        qgoal = True

rrt_tree = RRT(Node(sx, sy), Node(gx, gy))
rrt_star_tree = RRT_star(Node(sx, sy), Node(gx, gy))

rrt_path = get_final_path(rrt_tree[-1])
rrt_star_path = get_final_path(rrt_star_tree[-1])

gif_filename = 'simulation4_epsilon5.gif'
n_frames = max(len(rrt_tree), len(rrt_star_tree))

with imageio.get_writer(gif_filename, mode='I', fps=8) as writer:
    for i in range(0, n_frames):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), facecolor='#ececec')
        fig.subplots_adjust(left=0.05, right=0.95, top=0.88, bottom=0.08, wspace=0.12)
        show_path = (i == n_frames - 1) #true only in the last frame
        draw_panel(ax1, rrt_tree, rrt_path, TREE_COLOR_RRT, 'RRT', i, show_path)
        draw_panel(ax2, rrt_star_tree, rrt_star_path, TREE_COLOR_RRTS, 'RRT*', i, show_path)
        fig.suptitle(f'RRT vs RRT*', fontsize=14, fontweight='bold', color='#2c3e50', y=0.97)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=90)
        buf.seek(0)
        writer.append_data(imageio.imread(buf))
        plt.close()

/tmp/ipykernel_435/3714551241.py:32: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(buf))
